In [14]:
import pandas as pd
import re
import yaml

import psrcelmerpy

cfg = yaml.safe_load(open("configs/settings.yaml"))
e_conn = psrcelmerpy.ElmerConn()

In [15]:
data_dir = cfg["data_dir"]
county_map = cfg["county_map"]  # "King County" -> id
bare_county_map = {name.replace(" County", ""): name for name in county_map}  # "King" -> "King County"

# OFM sheet name -> component label; column headers span an April-to-April
# period (e.g. "1960-1961") and some Jurisdiction values carry footnote marks,
# so counties are matched on the plain "County" column instead.
OFM_SHEETS = {
    "Residual Net Migration": "Migration",
    "Natural Change": "Natural Increase",
}


def load_ofm_components(file_path):
    """Read the OFM components-of-change workbook's Natural Change and Residual
    Net Migration sheets by county into a long [region, component, year, value]
    frame, attributing each April-to-April period to its ending year to match
    the population estimate it feeds into."""
    frames = []
    for sheet_name, component in OFM_SHEETS.items():
        df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=3)
        df["region"] = df["County"].map(bare_county_map)
        df = df.loc[df["region"].notna()].copy()

        year_cols = [c for c in df.columns if re.search(r"\d{4}-\d{4}", str(c))]
        long = df.melt(id_vars="region", value_vars=year_cols, var_name="period", value_name="value")
        long["year"] = long["period"].str.extract(r"\d{4}-(\d{4})").astype(int)
        long["component"] = component
        frames.append(long[["region", "component", "year", "value"]])
    return pd.concat(frames, ignore_index=True)


In [16]:
import os

ofm_components = load_ofm_components(os.path.join(data_dir, cfg["ofm_components_file"]))

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
ofm_region_totals = ofm_components.groupby(["component", "year"], as_index=False)["value"].sum()
ofm_region_totals["region"] = "Region"
ofm_components = pd.concat([ofm_components, ofm_region_totals], ignore_index=True)

In [17]:
components_by_year = (
    ofm_components.loc[ofm_components["region"] == "Region"]
    .pivot(index="year", columns="component", values="value")
    .sort_index()
)

In [18]:
components_dir = cfg["remi_components_dir"]

COMPONENT_LABELS = {
    "Total Migrants - All Races": "Migration",
    "Natural Growth - All Races": "Natural Increase",
}


def load_remi_components(file_path):
    """Read a REMI components workbook's Migration and Natural Increase rows
    by county (in thousands) into a long [region, component, year, value] frame."""
    df = pd.read_excel(file_path, sheet_name="All", header=5)
    df = df.rename(columns={df.columns[0]: "region", df.columns[1]: "category"})
    year_cols = list(df.columns[3:])

    comp = df[df["region"].isin(county_map) & df["category"].isin(COMPONENT_LABELS)].copy()
    comp["component"] = comp["category"].map(COMPONENT_LABELS)

    long = comp.melt(
        id_vars=["region", "component"],
        value_vars=year_cols,
        var_name="year",
        value_name="value",
    )
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    return long


In [19]:
remi_frames = []
for forecast in cfg["remi_components_forecasts"]:
    file_path = os.path.join(components_dir, forecast["filename"])
    df = load_remi_components(file_path)
    df["name"] = forecast["name"]
    remi_frames.append(df)
remi_components = pd.concat(remi_frames, ignore_index=True)

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
remi_region_totals = remi_components.groupby(["name", "component", "year"], as_index=False)["value"].sum()
remi_region_totals["region"] = "Region"
remi_components = pd.concat([remi_components, remi_region_totals], ignore_index=True)


c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [20]:
remi_components_by_year = (
    remi_components.loc[remi_components["region"] == "Region"]
    .pivot_table(index="year", columns=["name", "component"], values="value")
    .sort_index()
)

In [21]:
output_dir = cfg["output_dir"]

# OFM data before 1990 is excluded; REMI and OFM become separate columns so overlapping years show both.
history_min_year = 1990
ofm_hist = ofm_components.loc[ofm_components["year"] >= history_min_year].rename(columns={"value": "OFM"})
history_max_year = int(ofm_hist["year"].max())
remi_wide = remi_components.drop(columns="name").rename(columns={"value": "REMI"})

comp_all = pd.merge(ofm_hist, remi_wide, on=["region", "component", "year"], how="outer").sort_values(
    ["region", "component", "year"]
)

comp_all.to_csv(os.path.join(output_dir, "components_history_forecast.csv"), index=False)

In [22]:
def load_ofm_population(file_path):
    """Read the OFM Population sheet's April 1 estimates by county into a long
    [region, year, value] frame; column headers end in a year, some with footnote marks."""
    df = pd.read_excel(file_path, sheet_name="Population", skiprows=3)
    df["region"] = df["County"].map(bare_county_map)
    df = df.loc[df["region"].notna()].copy()

    year_cols = [c for c in df.columns if re.search(r"\d{4}", str(c))]
    long = df.melt(id_vars="region", value_vars=year_cols, var_name="period", value_name="value")
    long["year"] = long["period"].str.extract(r"(\d{4})").astype(int)
    return long[["region", "year", "value"]]


ofm_population = load_ofm_population(os.path.join(data_dir, cfg["ofm_components_file"]))

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
ofm_population_region_totals = ofm_population.groupby("year", as_index=False)["value"].sum()
ofm_population_region_totals["region"] = "Region"
ofm_population = pd.concat([ofm_population, ofm_population_region_totals], ignore_index=True)

POPULATION_LABELS = {"Total Population": "Population"}


def load_remi_population(file_path):
    """Read a REMI components workbook's Total Population row by county (in
    thousands) into a long [region, year, value] frame."""
    df = pd.read_excel(file_path, sheet_name="All", header=5)
    df = df.rename(columns={df.columns[0]: "region", df.columns[1]: "category"})
    year_cols = list(df.columns[3:])

    pop = df[df["region"].isin(county_map) & df["category"].isin(POPULATION_LABELS)].copy()
    long = pop.melt(id_vars="region", value_vars=year_cols, var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    return long[["region", "year", "value"]]


remi_population_frames = []
for forecast in cfg["remi_components_forecasts"]:
    file_path = os.path.join(components_dir, forecast["filename"])
    df = load_remi_population(file_path)
    df["name"] = forecast["name"]
    remi_population_frames.append(df)
remi_population = pd.concat(remi_population_frames, ignore_index=True)

remi_population_region_totals = remi_population.groupby(["name", "year"], as_index=False)["value"].sum()
remi_population_region_totals["region"] = "Region"
remi_population = pd.concat([remi_population, remi_population_region_totals], ignore_index=True)



c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### Population totals chart

In [23]:
import plotly.express as px

remi_forecast_population = remi_population.loc[
    (remi_population["region"] == "Region") & (remi_population["year"] >= 2026)
].sort_values("year")
remi_forecast_population["value"] = (remi_forecast_population["value"] / 1_000_000).round(2)

fig = px.bar(remi_forecast_population, x="year", y="value", title="REMI Region Population Forecast")
fig.update_layout(xaxis_title="Year", yaxis_title="Total Population (Millions)")
fig.update_traces(hovertemplate="Year=%{x}<br>Total Population=%{y:.2f}M<extra></extra>")

# Callouts for the first and last forecast years.
for _, row in remi_forecast_population.iloc[[0, -1]].iterrows():
    fig.add_annotation(
        x=row["year"],
        y=row["value"],
        text=f"{row['value']:.2f}M",
        showarrow=False,
        yshift=12,
    )

fig.show()

In [24]:
EMPLOYMENT_PREFIX = "Employment - "


def load_remi_employment(file_path):
    """Read a REMI workbook's industry-level Employment rows (excludes the
    separate Employment by Occupation breakdown) and sum them by county (in
    thousands) into a long [region, year, value] frame."""
    df = pd.read_excel(file_path, sheet_name="All", header=5)
    df = df.rename(columns={df.columns[0]: "region", df.columns[1]: "category"})
    year_cols = list(df.columns[3:])

    emp = df[df["region"].isin(county_map) & df["category"].str.startswith(EMPLOYMENT_PREFIX)].copy()
    long = emp.melt(id_vars="region", value_vars=year_cols, var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    return long.groupby(["region", "year"], as_index=False)["value"].sum()


remi_dir = cfg["remi_dir"]

remi_employment_frames = []
for forecast in cfg["remi_forecasts"]:
    file_path = os.path.join(remi_dir, forecast["filename"])
    df = load_remi_employment(file_path)
    df["name"] = forecast["name"]
    remi_employment_frames.append(df)
remi_employment = pd.concat(remi_employment_frames, ignore_index=True)

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
remi_employment_region_totals = remi_employment.groupby(["name", "year"], as_index=False)["value"].sum()
remi_employment_region_totals["region"] = "Region"
remi_employment = pd.concat([remi_employment, remi_employment_region_totals], ignore_index=True)


c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### Job totals chart

In [25]:
remi_forecast_employment = remi_employment.loc[
    (remi_employment["region"] == "Region") & (remi_employment["year"] >= cfg['remi_base_year'])
].sort_values("year")
remi_forecast_employment["value"] = (remi_forecast_employment["value"] / 1_000_000).round(2)

fig = px.bar(remi_forecast_employment, x="year", y="value", title="REMI Region Employment Forecast")
fig.update_layout(xaxis_title="Year", yaxis_title="Total Employment (Millions)")
fig.update_traces(marker_color="#00cc96", hovertemplate="Year=%{x}<br>Total Employment=%{y:.2f}M<extra></extra>")

# Callouts for the first and last forecast years.
for _, row in remi_forecast_employment.iloc[[0, -1]].iterrows():
    fig.add_annotation(
        x=row["year"],
        y=row["value"],
        text=f"{row['value']:.2f}M",
        showarrow=False,
        yshift=12,
    )

fig.show()


In [26]:
QCEW_NAICS_GROUPS = {
    "naics_3133": ["naics_31", "naics_32", "naics_33"],
    "naics_4445": ["naics_44", "naics_45"],
    "naics_4849": ["naics_48", "naics_49"],
}


def load_qcew(qcew_year):
    """Read covered employment by 2-digit NAICS per county for a year, combining
    NAICS groups 31-33, 44-45 and 48-49 and folding "Education" into "Government"
    (this source tracks public school district staff separately) to match REMI's
    employment categories, into a long [region, naics, year, value] frame matching
    remi_employment_naics."""
    df = e_conn.get_table(schema='employment_summaries', table_name='covered_employment_by_city_by_naics2')
    df = df.loc[(df['data_year'] == qcew_year) & (df['City'] == 'Total') & (df['County'] != 'Region')].copy()
    df = df.drop(columns=['data_year', 'City'])
    supressed_cols = [col for col in df.columns if 'suppressed' in col]
    df = df.drop(columns=supressed_cols)
    df["Government"] = df["Government"] + df["Education"]
    df = df.drop(columns="Education")

    before_total = df.drop(columns=['County', 'Total']).sum(axis=1)
    for combined_col, source_cols in QCEW_NAICS_GROUPS.items():
        df[combined_col] = df[source_cols].sum(axis=1)
    df = df.drop(columns=[col for cols in QCEW_NAICS_GROUPS.values() for col in cols])

    # Confirm combining the NAICS groups didn't change the overall total, i.e. no industry was missed.
    after_total = df.drop(columns=['County', 'Total']).sum(axis=1)
    assert (before_total - after_total).abs().max() < 1, "Combining NAICS groups should not change the total"

    df["region"] = df["County"].map(bare_county_map)
    naics_cols = [col for col in df.columns if col.startswith("naics_") or col == "Government"]
    long = df.melt(id_vars="region", value_vars=naics_cols, var_name="naics", value_name="value")
    long["year"] = qcew_year
    return long[["region", "naics", "year", "value"]]


qcew_year = 2025
qcew_df = load_qcew(qcew_year)

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
qcew_region_totals = qcew_df.groupby(["naics", "year"], as_index=False)["value"].sum()
qcew_region_totals["region"] = "Region"
qcew_df = pd.concat([qcew_df, qcew_region_totals], ignore_index=True)


REMI_INDUSTRY_LABELS = {
    "Forestry, fishing, and hunting": "naics_11",
    "Mining": "naics_21",
    "Utilities": "naics_22",
    "Construction": "naics_23",
    "Manufacturing": "naics_3133",
    "Wholesale trade": "naics_42",
    "Retail trade": "naics_4445",
    "Transportation and warehousing": "naics_4849",
    "Information": "naics_51",
    "Finance and insurance": "naics_52",
    "Real estate and rental and leasing": "naics_53",
    "Professional, scientific, and technical services": "naics_54",
    "Management of companies and enterprises": "naics_55",
    "Administrative, support, waste management, and remediation services": "naics_56",
    "Educational services; private": "naics_61",
    "Health care and social assistance": "naics_62",
    "Arts, entertainment, and recreation": "naics_71",
    "Accommodation and food services": "naics_72",
    "Other services (except public administration)": "naics_81",
    "State and Local Government": "Government",
    "Federal Civilian": "Government",
    "Federal Military": "Military",
    # "Farm" is intentionally left unmapped so it gets dropped below.
}


def load_remi_employment_naics(file_path):
    """Read a REMI workbook's Employment-by-industry rows by county (in
    thousands) into a long [region, naics, year, value] frame; Federal Civilian
    and State and Local Government are combined into "Government" (QCEW's own
    Government count is likewise ESD covered employment combined with a survey
    of federal employment), Farm is dropped, and grouped NAICS codes (31-33,
    44-45, 48-49) keep REMI's combined value under a combined column name."""
    df = pd.read_excel(file_path, sheet_name="All", header=5)
    df = df.rename(columns={df.columns[0]: "region", df.columns[1]: "category"})
    year_cols = list(df.columns[3:])

    industry_label = df["category"].str.extract(r"^Employment - (.+)$", expand=False)
    emp = df.loc[df["region"].isin(county_map) & industry_label.notna()].copy()
    emp["naics"] = industry_label.loc[emp.index].map(REMI_INDUSTRY_LABELS)
    emp = emp.loc[emp["naics"].notna()].copy()

    long = emp.melt(id_vars=["region", "naics"], value_vars=year_cols, var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    # Government combines two source rows per county/year, so re-aggregate after mapping.
    return long.groupby(["region", "naics", "year"], as_index=False)["value"].sum()


remi_employment_naics_frames = []
for forecast in cfg["remi_forecasts"]:
    emp_file_path = os.path.join(remi_dir, forecast["filename"])
    emp_df = load_remi_employment_naics(emp_file_path)
    emp_df["name"] = forecast["name"]
    remi_employment_naics_frames.append(emp_df)
remi_employment_naics = pd.concat(remi_employment_naics_frames, ignore_index=True)

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
remi_employment_naics_region_totals = remi_employment_naics.groupby(["name", "naics", "year"], as_index=False)["value"].sum()
remi_employment_naics_region_totals["region"] = "Region"
remi_employment_naics = pd.concat([remi_employment_naics, remi_employment_naics_region_totals], ignore_index=True)


def remi_employment_naics_totals_check(file_path):
    """Sum every REMI Employment-by-industry row (including Farm) by year, to
    confirm Farm is the only category excluded from remi_employment_naics."""
    raw = pd.read_excel(file_path, sheet_name="All", header=5)
    raw = raw.rename(columns={raw.columns[0]: "region", raw.columns[1]: "category"})
    year_cols = list(raw.columns[3:])

    industry_rows = raw.loc[raw["region"].isin(county_map) & raw["category"].str.match(r"^Employment - ")]
    all_industry_total = industry_rows[year_cols].sum() * 1000
    farm_total = industry_rows.loc[industry_rows["category"] == "Employment - Farm", year_cols].sum() * 1000
    all_industry_total.index = all_industry_total.index.astype(int)
    farm_total.index = farm_total.index.astype(int)
    return all_industry_total, farm_total


_remi_employment_file = os.path.join(remi_dir, cfg["remi_forecasts"][0]["filename"])
_all_industry_total, _farm_total = remi_employment_naics_totals_check(_remi_employment_file)
_mapped_total = remi_employment_naics.loc[remi_employment_naics["region"] != "Region"].groupby("year")["value"].sum()

_totals_check = (_all_industry_total - _farm_total - _mapped_total).round(2)
assert (_totals_check.abs() < 1).all(), "Mapped employment should equal REMI's total minus Farm"


remi_employment_naics = remi_employment_naics.sort_values(["name", "region", "naics", "year"])
remi_employment_naics["growth_rate"] = remi_employment_naics.groupby(["name", "region", "naics"])["value"].pct_change()
# Military has no QCEW-covered counterpart, so set it aside to rejoin as REMI's actual values.
military_naics = remi_employment_naics.loc[remi_employment_naics["naics"] == "Military"].copy()
remi_employment_naics = remi_employment_naics.loc[remi_employment_naics["naics"] != "Military"].copy()

c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### QCEW covered-employment forecast

In [27]:
# REMI's employment figure includes proprietors, which QCEW's covered employment excludes and
# REMI doesn't report separately, so the gap is captured empirically as a ratio at the one year
# both sources cover, then held constant and applied to REMI's forecast.
remi_anchor = remi_employment_naics.loc[
    (remi_employment_naics["name"] == "REMI") & (remi_employment_naics["year"] == qcew_year), ["region", "naics", "value"]
].rename(columns={"value": "remi_value"})
qcew_anchor = qcew_df[["region", "naics", "value"]].rename(columns={"value": "qcew_value"})

qcew_ratio = pd.merge(qcew_anchor, remi_anchor, on=["region", "naics"], how="inner")
assert len(qcew_ratio) == len(qcew_anchor) == len(remi_anchor), "Every QCEW/REMI region-industry combo should have a match"
qcew_ratio["ratio"] = qcew_ratio["qcew_value"] / qcew_ratio["remi_value"]

# Sanity check: ratio should be near 1 for Government (fully covered) and lower for
# industries with more self-employment/proprietors (e.g. real estate, professional services).
qcew_ratio.loc[qcew_ratio["region"] == "Region"].sort_values("ratio")


qcew_forecast = pd.merge(
    remi_employment_naics.loc[remi_employment_naics["name"] == "REMI", ["region", "naics", "year", "value"]],
    qcew_ratio[["region", "naics", "ratio"]],
    on=["region", "naics"],
    how="inner",
)
qcew_forecast["value"] = qcew_forecast["value"] * qcew_forecast["ratio"]
qcew_forecast = qcew_forecast.drop(columns="ratio")

# At the anchor year the scaled forecast should reproduce QCEW's actual value exactly, by construction.
_anchor_check = pd.merge(
    qcew_forecast.loc[qcew_forecast["year"] == qcew_year], qcew_df, on=["region", "naics", "year"], suffixes=("_scaled", "_actual")
)
assert (_anchor_check["value_scaled"] - _anchor_check["value_actual"]).abs().max() < 1e-6

# Military passes through as REMI's own value, since QCEW has no counterpart to scale against.
military_actual = military_naics.loc[military_naics["name"] == "REMI", ["region", "naics", "year", "value"]]

# QCEW is the actual, observed value at the anchor year; REMI (scaled to QCEW terms, or passed through) supplies every other year.
qcew_employment_all = pd.concat(
    [
        qcew_df.assign(source="QCEW"),
        qcew_forecast.loc[qcew_forecast["year"] != qcew_year].assign(source="REMI (QCEW terms)"),
        military_actual.assign(source="REMI (actual)"),
    ],
    ignore_index=True,
).sort_values(["region", "naics", "year"])


qcew_forecast.to_csv(os.path.join(output_dir, "REMI_on_QCEW_forecast.csv"))

#### Covered-employment forecast chart

In [ ]:
qcew_employment_forecast = (
    qcew_employment_all.loc[(qcew_employment_all["region"] == "Region") & (qcew_employment_all["year"] > qcew_year)]
    .groupby("year", as_index=False)["value"]
    .sum()
    .sort_values("year")
)
qcew_employment_forecast["value"] = (qcew_employment_forecast["value"] / 1_000_000).round(2)

fig = px.bar(qcew_employment_forecast, x="year", y="value", title="REMI-on-QCEW Region Employment Forecast")
fig.update_layout(xaxis_title="Year", yaxis_title="Total Employment (Millions)")
fig.update_traces(hovertemplate="Year=%{x}<br>Total Employment=%{y:.2f}M<extra></extra>")

# Callouts for the first and last forecast years.
for _, row in qcew_employment_forecast.iloc[[0, -1]].iterrows():
    fig.add_annotation(
        x=row["year"],
        y=row["value"],
        text=f"{row['value']:.2f}M",
        showarrow=False,
        yshift=12,
    )

fig.show()

#### Population growth table

In [86]:
remi_base_year = cfg["remi_base_year"]

periods = [
    ("OFM 1990s", 1990, 1999),
    ("OFM 2000s", 2000, 2009),
    ("OFM 2010s", 2010, 2019),
    (f"OFM 2020-{history_max_year}", 2020, history_max_year),
    (f"REMI {remi_base_year + 1}-2060", remi_base_year + 1, 2060),
]

def period_avg(df, start, end):
    subset = df.loc[(df["region"] == "Region") & (df["year"] >= start) & (df["year"] <= end)]
    return subset.groupby("component")["value"].mean()

# Historical periods use OFM only, the forecast period uses REMI only.
ofm_periods, remi_period = periods[:4], periods[4]
ofm_avg_table = pd.concat(
    [period_avg(ofm_components, start, end).rename(label) for label, start, end in ofm_periods], axis=1
).T
remi_label, remi_start, remi_end = remi_period
remi_avg_table = period_avg(remi_components, remi_start, remi_end).rename(remi_label).to_frame().T


ofm_pop_region = ofm_population.loc[ofm_population["region"] == "Region"].sort_values("year").set_index("year")["value"]
ofm_growth = ofm_pop_region.diff()

remi_pop_region = remi_population.loc[remi_population["region"] == "Region"].sort_values("year").set_index("year")["value"]
remi_growth = remi_pop_region.diff()


def growth_period_avg(growth, start, end):
    return growth.loc[(growth.index >= start) & (growth.index <= end)].mean()


# Historical periods use OFM only, the forecast period uses REMI only, same split as decade_avg_table.
ofm_growth_avg = pd.Series(
    {label: growth_period_avg(ofm_growth, start, end) for label, start, end in ofm_periods}, name="Avg. Annual Pop Growth"
)
remi_growth_avg = pd.Series({remi_label: growth_period_avg(remi_growth, remi_start, remi_end)}, name="Avg. Annual Pop Growth")

growth_avg_table = pd.concat([ofm_growth_avg, remi_growth_avg]).to_frame().round(0).astype(int)
growth_avg_table = growth_avg_table.rename_axis(index="Period", columns=None)
growth_avg_table.style.format({
    'Avg. Annual Pop Growth': '{:,.0f}'
})

,Avg. Annual Pop Growth
Period,
OFM 1990s,"55,536"
OFM 2000s,"44,341"
OFM 2010s,"55,711"
OFM 2020-2026,"48,978"
REMI 2025-2060,"47,714"


#### Population components table

In [87]:
decade_avg_table = pd.concat([ofm_avg_table, remi_avg_table]).round(0).astype(int)
decade_avg_table = decade_avg_table.rename_axis(index="Period", columns=None)
decade_avg_table.style.format({
    'Migration': '{:,.0f}',
    'Natural Increase': '{:,.0f}'
})

,Migration,Natural Increase
Period,,
OFM 1990s,"32,325","23,211"
OFM 2000s,"21,751","22,589"
OFM 2010s,"32,712","22,999"
OFM 2020-2026,"34,123","14,855"
REMI 2025-2060,"37,572","10,142"


#### Job growth table

In [124]:
job_periods = [
    ("2000s", 2000, 2009),
    ("2010s", 2010, 2019),
    ("2020-2024", 2020, 2024),
    ("2025-2060", 2025, 2060),
]

def job_growth_period_avg(df, start, end):
    subset = df.loc[(df["region"] == "Region") & (df["year"] >= start) & (df["year"] <= end)]
    return subset["growth"].mean()

job_growth_avg_table = pd.Series(
    {label: job_growth_period_avg(remi_employment, start, end) for label, start, end in job_periods},
    name="Avg. Annual Employment Growth",
).round(0).astype(int).to_frame()
job_growth_avg_table = job_growth_avg_table.rename_axis(index="Period", columns=None)
job_growth_avg_table.style.format({"Avg. Annual Employment Growth": "{:,.0f}"})


,Avg. Annual Employment Growth
Period,
2000s,"19,076"
2010s,"51,904"
2020-2024,"21,585"
2025-2060,"30,784"


In [128]:
def load_remi_national_employment(file_path):
    """Read a REMI national workbook's industry-level Employment rows (excludes
    the separate Employment by Occupation breakdown) summed into a long
    [region, year, value] frame for the Nation."""
    df = pd.read_excel(file_path, sheet_name="All", header=5)
    df = df.rename(columns={df.columns[0]: "region", df.columns[1]: "category"})
    year_cols = list(df.columns[3:])

    emp = df[(df["region"] == "Nation") & df["category"].str.startswith(EMPLOYMENT_PREFIX)].copy()
    long = emp.melt(id_vars="region", value_vars=year_cols, var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    return long.groupby(["region", "year"], as_index=False)["value"].sum()


national_file_path = os.path.join(remi_dir, cfg["remi_national_forecasts"])
remi_national_employment = load_remi_national_employment(national_file_path).sort_values("year")
remi_national_employment["growth"] = remi_national_employment["value"].diff()


c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### Job growth rates compared to nation

In [130]:
def cagr(df, region, start, end):
    subset = df.loc[(df["region"] == region) & (df["year"] >= start) & (df["year"] <= end)].sort_values("year")
    start_value, end_value = subset["value"].iloc[0], subset["value"].iloc[-1]
    n_years = subset["year"].iloc[-1] - subset["year"].iloc[0]
    return (end_value / start_value) ** (1 / n_years) - 1


cagr_table = pd.DataFrame(
    {
        "PSRC Region": {label: cagr(remi_employment, "Region", start, end) for label, start, end in job_periods},
        "Nation": {label: cagr(remi_national_employment, "Nation", start, end) for label, start, end in job_periods},
    }
)
cagr_table = cagr_table.rename_axis(index="Period", columns=None)
cagr_table.style.format("{:.2%}")


,PSRC Region,Nation
Period,,
2000s,0.86%,0.60%
2010s,2.43%,1.72%
2020-2024,1.98%,2.81%
2025-2060,0.87%,0.35%


#### Information industry growth rates compared to nation

In [131]:
def load_remi_industry(file_path, category, regions):
    """Read a single Employment industry category from a REMI workbook into a
    long [region, year, value] frame."""
    df = pd.read_excel(file_path, sheet_name="All", header=5)
    df = df.rename(columns={df.columns[0]: "region", df.columns[1]: "category"})
    year_cols = list(df.columns[3:])

    ind = df[df["region"].isin(regions) & (df["category"] == category)].copy()
    long = ind.melt(id_vars="region", value_vars=year_cols, var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    return long[["region", "year", "value"]]


def industry_cagr_table(category):
    """Build a PSRC Region vs. Nation CAGR table (same periods as cagr_table) for one REMI industry category."""
    regional_frames = []
    for forecast in cfg["remi_forecasts"]:
        file_path = os.path.join(remi_dir, forecast["filename"])
        regional_frames.append(load_remi_industry(file_path, category, county_map))
    regional = pd.concat(regional_frames, ignore_index=True)
    region_totals = regional.groupby("year", as_index=False)["value"].sum()
    region_totals["region"] = "Region"
    regional = pd.concat([regional, region_totals], ignore_index=True)

    national = load_remi_industry(national_file_path, category, ["Nation"])

    table = pd.DataFrame(
        {
            "PSRC Region": {label: cagr(regional, "Region", start, end) for label, start, end in job_periods},
            "Nation": {label: cagr(national, "Nation", start, end) for label, start, end in job_periods},
        }
    )
    return table.rename_axis(index="Period", columns=None)


information_cagr_table = industry_cagr_table("Employment - Information")
information_cagr_table.style.format("{:.2%}")


c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,PSRC Region,Nation
Period,,
2000s,1.10%,-2.48%
2010s,4.17%,0.56%
2020-2024,2.01%,3.43%
2025-2060,0.79%,0.44%


#### Professional, scientific, and technical services growth rates compared to nation

In [132]:
professional_scientific_cagr_table = industry_cagr_table("Employment - Professional, scientific, and technical services")
professional_scientific_cagr_table.style.format("{:.2%}")

c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jkolberg\PythonProjects\PSRC\macro-explorer\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,PSRC Region,Nation
Period,,
2000s,1.91%,1.66%
2010s,2.68%,2.36%
2020-2024,3.20%,3.29%
2025-2060,1.31%,0.84%


#### Forecast comparison table

In [149]:
def load_ofm_forecast_population(file_path):
    """Read the OFM GMA growth-management forecast's Population sheet, keeping
    the age="Total" row per county, into a long [region, year, value] frame.
    Data points are unevenly spaced (2020, 2022, then every 5 years); the
    first "Projection"-labeled year marks the start of the forecast."""
    raw = pd.read_excel(file_path, sheet_name="Population", header=None)
    labels, years = raw.iloc[0], raw.iloc[1]
    year_cols = {int(year): idx for idx, year in years.items() if pd.notna(year)}
    forecast_start_year = int(years[labels == "Projection"].iloc[0])

    df = raw.iloc[3:].copy()
    df["region"] = df[0].map(bare_county_map)
    df = df.loc[df["region"].notna() & (df[1] == "Total")]

    records = []
    for year, col_idx in year_cols.items():
        total_col = col_idx + 2
        sub = df[["region", total_col]].rename(columns={total_col: "value"})
        sub["year"] = year
        records.append(sub)
    long = pd.concat(records, ignore_index=True)
    long["value"] = long["value"].astype(float)
    return long[["region", "year", "value"]], forecast_start_year


ofm_forecast_population, ofm_forecast_start_year = load_ofm_forecast_population(
    os.path.join(data_dir, cfg["ofm_forecast_file"])
)

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
ofm_forecast_population_region_totals = ofm_forecast_population.groupby("year", as_index=False)["value"].sum()
ofm_forecast_population_region_totals["region"] = "Region"
ofm_forecast_population = pd.concat(
    [ofm_forecast_population, ofm_forecast_population_region_totals], ignore_index=True
)


In [150]:
woods_poole_dir = cfg["woods_poole_forecast_dir"]
county_id_map = {v: k for k, v in county_map.items()}  # id -> "King County"


def load_woods_poole_population(dir_path, files):
    """Read each Woods & Poole county CSV's Total Population row (in
    thousands) into a long [region, year, value] frame."""
    frames = []
    for entry in files:
        (county_id, filename), = entry.items()
        raw = pd.read_csv(os.path.join(dir_path, filename), skiprows=2, index_col=0)
        pop = raw.loc["TOTAL POPULATION (in thousands)"].rename("value").reset_index()
        pop = pop.rename(columns={"index": "year"})
        pop["year"] = pop["year"].astype(int)
        pop["value"] = pop["value"].astype(float) * 1000
        pop["region"] = county_id_map[county_id]
        frames.append(pop)
    return pd.concat(frames, ignore_index=True)


woods_poole_population = load_woods_poole_population(woods_poole_dir, cfg["woods_poole_files"])

# "Region" is the sum of the four counties, computed here rather than relying on a workbook total row.
woods_poole_region_totals = woods_poole_population.groupby("year", as_index=False)["value"].sum()
woods_poole_region_totals["region"] = "Region"
woods_poole_population = pd.concat([woods_poole_population, woods_poole_region_totals], ignore_index=True)


In [151]:
def load_luvit_population(file_path):
    """Read the LUVit CSV's Region-level Total Population row (in thousands,
    already summed across counties) into a long [region, year, value] frame."""
    df = pd.read_csv(file_path)
    pop = df.loc[
        (df["Main Measure"] == "Population")
        & (df["Detailed Measure"] == "Total Population")
        & (df["Region"] == "Region")
    ]
    year_cols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c))]
    long = pop.melt(value_vars=year_cols, var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = long["value"].astype(float) * 1000
    long["region"] = "Region"
    return long[["region", "year", "value"]]


luvit_population = load_luvit_population(os.path.join(data_dir, cfg["luvit_forecast_file"]))


In [158]:
def avg_annual_change(df, region, start_year):
    """Average annual change in value for a region from the first year at or
    after start_year through the series' last year, using only the endpoints
    (handles unevenly-spaced data such as the OFM GMA forecast)."""
    subset = df.loc[(df["region"] == region) & (df["year"] >= start_year)].sort_values("year")
    start_value, end_value = subset["value"].iloc[0], subset["value"].iloc[-1]
    n_years = subset["year"].iloc[-1] - subset["year"].iloc[0]
    return (end_value - start_value) / n_years


forecast_comparison_table = pd.Series(
    {
        "LUV-it (2018)": avg_annual_change(luvit_population, "Region", cfg["luvit_forecast_start_year"]),
        "OFM GMA (2022)": avg_annual_change(ofm_forecast_population, "Region", ofm_forecast_start_year),
        "REMI (2026)": avg_annual_change(remi_population, "Region", cfg["remi_base_year"] + 1),
        "Woods & Poole (2026)": avg_annual_change(
            woods_poole_population, "Region", cfg["woods_poole_forecast_start_year"]
        ),
        
    },
    name="Avg. Annual Population Growth",
).round(0).astype(int).to_frame()
forecast_comparison_table = forecast_comparison_table.rename_axis(index="Forecast", columns=None)
forecast_comparison_table.style.format({"Avg. Annual Population Growth": "{:,.0f}"})

,Avg. Annual Population Growth
Forecast,
LUV-it (2018),"54,719"
OFM GMA (2022),"41,065"
REMI (2026),"46,009"
Woods & Poole (2026),"40,165"
